### ELECTRA fine-tuning on CoLA

This notebook contains pipeline for fine-tuning ELECTRA model on CoLA and extended versions of CoLA, generated in `chatgpt_scoring` notebook.

In [2]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import transformers as ts
import datasets
import torch
from torch import nn

In [ ]:
# Download cola_e_balanced.csv
# This file contains CoLA dataset merged with generated data, filtered by ChatGPT scores of fluency.
# (If generated label matched ChatGPT label, we added sample to dataset.)
# We added all negative samples and 2k of positive ones, to make data more balanced.
!gdown 1xwSOFmcK3HLjqEmNezMSZrcGzbADBsdh

# Download cola_ecl_balanced.csv
# This file contains CoLA dataset merged with generated data, labeled by ChatGPT scores of fluency
# (If generated label did not match ChatGPT label, we changed label)
# We added all negative samples and 4k of positive ones, to make data more balanced
!gdown 1qCcsSnwZ9ef7S8vRVXKF6MMj9_LpN7T1

In [ ]:
# compute_metrics, tokenizer and data_collator setup
accuracy_metric = datasets.load_metric('accuracy', cache_dir='tmp/metrics')
cola_metric = datasets.load_metric('glue', 'cola', cache_dir='tmp/metrics')

def compute_metrics(outputs):
  logits, labels = outputs
  preds = logits.argmax(1)

  metrics_dict = accuracy_metric.compute(
      references=labels, predictions=preds)
  metrics_dict.update(cola_metric.compute(
      references=labels, predictions=preds))

  return metrics_dict

# Upload the tokenizer
tokenizer = ts.AutoTokenizer.from_pretrained('google/electra-large-discriminator', cache_dir='tmp/tokenizer')

data_collator = ts.DataCollatorWithPadding(tokenizer=tokenizer)

In [4]:
# Prepare train data from the dataset of your choice. 
# Validation data is always from CoLA validation subset.
# mode is either 'cola', 'extended' or 'extended_chatgpt'
def prepare_setup(mode='cola'): 
  # Upload the data
  if mode == 'extended':
    data = datasets.load_dataset("csv", data_files="cola_e_balanced.csv")
  else:
    data = datasets.load_dataset("csv", data_files="cola_ecl_balanced.csv")
  data_cola = datasets.load_dataset('glue', 'cola', cache_dir='tmp/data')

  # tokenize train data
  if mode != 'cola':
    def tokenizing_fn(instance):
        return tokenizer(instance['text'], truncation=True)
    tokenized_train = data["train"].map(tokenizing_fn, batched=True)
    tokenized_train = tokenized_train.remove_columns(['text', 'id'])
  else:
    def tokenizing_fn(instance):
        return tokenizer(instance['sentence'], truncation=True)
    tokenized_train = data_cola["train"].map(tokenizing_fn, batched=True)
    tokenized_train = tokenized_train.remove_columns(['sentence', 'idx'])

  # tokenize validation data
  def tokenizing_fn_val(instance):
      return tokenizer(instance['sentence'], truncation=True)
  tokenized_val = data_cola['validation'].map(tokenizing_fn_val, batched=True)
  tokenized_val = tokenized_val.remove_columns(['sentence', 'idx'])

  return tokenized_train, tokenized_val

In [ ]:
# prepare data for fine-tuning only on CoLA 
tokenized_train_cola, tokenized_val_cola = prepare_setup(mode='cola')
tokenized_train_cola

In [ ]:
# prepare data for fine-tuning on CoLA with generated data filtered by ChatGPT 
tokenized_train_ext, tokenized_val_ext = prepare_setup(mode='extended')
tokenized_train_ext

In [ ]:
# prepare data for fine-tuning on CoLA with generated data labeled by ChatGPT 
tokenized_train_ext_chatgpt, tokenized_val_ext_chatgpt = prepare_setup(mode='extended_chatgpt')
tokenized_train_ext_chatgpt

In [ ]:
# prepare model and training setup for fine-tuning only on CoLA 
model_cola = ts.AutoModelForSequenceClassification.from_pretrained(
    'google/electra-large-discriminator', cache_dir='tmp/model_cola', num_labels=2, classifier_dropout=0.1)

training_args_cola = ts.TrainingArguments(
    output_dir='tmp/model_output',
    # Batch size args
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=150,
    # Optimizer args
    learning_rate=1e-5,
    weight_decay=0,
    # Scheduler args
    # warmup_ratio=0.1,
    # Eval args
    metric_for_best_model='eval_accuracy',
    load_best_model_at_end=True,
    evaluation_strategy='epoch',
    logging_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    # WANDB args
    report_to="wandb",  # enable logging to W&B
)
callbacks = [ts.EarlyStoppingCallback(3)]

trainer_cola = ts.Trainer(
    model=model_cola,
    args=training_args_cola,
    data_collator=data_collator,
    train_dataset=tokenized_train_cola,
    eval_dataset=tokenized_val_cola,
    callbacks=callbacks,
    compute_metrics=compute_metrics
)

In [ ]:
# prepare model and training setup for fine-tuning on CoLA with generated data filtered by ChatGPT  
model_ext = ts.AutoModelForSequenceClassification.from_pretrained(
    'google/electra-large-discriminator', cache_dir='tmp/model_ext', num_labels=2, classifier_dropout=0.1)

training_args_ext = ts.TrainingArguments(
    output_dir='tmp/model_output',
    # Batch size args
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=150,
    # Optimizer args
    learning_rate=2e-5,
    weight_decay=0,
    # Scheduler args
    # warmup_ratio=0.1,
    # Eval args
    metric_for_best_model='eval_accuracy',
    load_best_model_at_end=True,
    evaluation_strategy='epoch',
    logging_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    # WANDB args
    report_to="wandb",  # enable logging to W&B
)
callbacks = [ts.EarlyStoppingCallback(3)]

trainer_ext = ts.Trainer(
    model=model_ext,
    args=training_args_ext,
    data_collator=data_collator,
    train_dataset=tokenized_train_ext,
    eval_dataset=tokenized_val_ext,
    callbacks=callbacks,
    compute_metrics=compute_metrics
)

In [ ]:
# prepare model and training setup for fine-tuning on CoLA with generated data labeled by ChatGPT 
model_ext_chatgpt = ts.AutoModelForSequenceClassification.from_pretrained(
    'google/electra-large-discriminator', cache_dir='tmp/model_ext_chatgpt', num_labels=2, classifier_dropout=0.1)

training_args_ext_chatgpt = ts.TrainingArguments(
    output_dir='tmp/model_output',
    # Batch size args
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=150,
    # Optimizer args
    learning_rate=1e-5,
    weight_decay=0,
    # Scheduler args
    # warmup_ratio=0.1,
    # Eval args
    metric_for_best_model='eval_accuracy',
    load_best_model_at_end=True,
    evaluation_strategy='epoch',
    logging_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    # WANDB args
    report_to="wandb",  # enable logging to W&B
)
callbacks = [ts.EarlyStoppingCallback(3)]

trainer_ext_chatgpt = ts.Trainer(
    model=model_ext_chatgpt,
    args=training_args_ext_chatgpt,
    data_collator=data_collator,
    train_dataset=tokenized_train_ext_chatgpt,
    eval_dataset=tokenized_val_ext_chatgpt,
    callbacks=callbacks,
    compute_metrics=compute_metrics
)

In [ ]:
!wandb login 

In [14]:
# fine-tuning only on CoLA 
trainer_cola.train()

/opt/conda/lib/python3.10/site-packages/transformers/optimization.py:407: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Matthews Correlation
1,0.405100,0.361499,0.856184,0.650410
2,0.232300,0.369480,0.872483,0.693070
3,0.142200,0.395841,0.882071,0.718251
4,0.096200,0.484925,0.881112,0.714295
5,0.068700,0.500145,0.881112,0.715064


TrainOutput(global_step=1340, training_loss=0.18891719561904224, metrics={'train_runtime': 302.9895, 'train_samples_per_second': 141.111, 'train_steps_per_second': 4.423, 'total_flos': 1805183813978328.0, 'train_loss': 0.18891719561904224, 'epoch': 5.0})

In [15]:
trainer_cola.evaluate()

{'eval_loss': 0.3958406150341034,
 'eval_accuracy': 0.8820709491850431,
 'eval_matthews_correlation': 0.7182514511816681,
 'eval_runtime': 1.2291,
 'eval_samples_per_second': 848.614,
 'eval_steps_per_second': 5.695,
 'epoch': 5.0}

----

In [18]:
# fine-tuning on CoLA with generated data filtered by ChatGPT 
trainer_ext.train()

/opt/conda/lib/python3.10/site-packages/transformers/optimization.py:407: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Matthews Correlation
1,0.435000,0.336841,0.862895,0.675448
2,0.239500,0.394987,0.863854,0.671145
3,0.124300,0.589529,0.855225,0.651400
4,0.072000,0.738376,0.859060,0.658181
5,0.041600,0.720397,0.867689,0.681938


TrainOutput(global_step=2315, training_loss=0.18247996056311588, metrics={'train_runtime': 428.649, 'train_samples_per_second': 172.822, 'train_steps_per_second': 5.401, 'total_flos': 3490548921345024.0, 'train_loss': 0.18247996056311588, 'epoch': 5.0})

In [13]:
trainer_ext.evaluate()

{'eval_loss': 0.6445770859718323,
 'eval_accuracy': 0.8744007670182167,
 'eval_matthews_correlation': 0.6972733290413318,
 'eval_runtime': 1.2468,
 'eval_samples_per_second': 836.528,
 'eval_steps_per_second': 5.614,
 'epoch': 5.0}

---

In [14]:
# fine-tuning on CoLA with generated data labeled by ChatGPT 
trainer_ext_chatgpt.train()

/opt/conda/lib/python3.10/site-packages/transformers/optimization.py:407: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Matthews Correlation
1,0.444900,0.351076,0.860019,0.664232
2,0.275800,0.375452,0.861937,0.671155
3,0.170300,0.461454,0.870566,0.689074
4,0.113000,0.526007,0.867689,0.681938
5,0.073100,0.615026,0.864813,0.673248


TrainOutput(global_step=3245, training_loss=0.21543833140415844, metrics={'train_runtime': 601.0112, 'train_samples_per_second': 172.759, 'train_steps_per_second': 5.399, 'total_flos': 4970954655203544.0, 'train_loss': 0.21543833140415844, 'epoch': 5.0})

In [15]:
trainer_ext_chatgpt.evaluate()

{'eval_loss': 0.46145448088645935,
 'eval_accuracy': 0.8705656759348035,
 'eval_matthews_correlation': 0.6890740780640155,
 'eval_runtime': 1.2353,
 'eval_samples_per_second': 844.334,
 'eval_steps_per_second': 5.667,
 'epoch': 5.0}

---

Upload models to HuggingFace:

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
model_cola.push_to_hub("grenlayk/electra-large-cola", create_pr=1)
model_ext.push_to_hub('grenlayk/electra-large-cola-extended', create_pr=1)
model_ext_chatgpt.push_to_hub('grenlayk/electra-large-cola-extended-chatgpt', create_pr=1)